Inspecting duration statistics for UiS4ADL before and after trimming and PAAL ADL.

In [ ]:
import os
import json
import warnings
import pandas as pd

# Suppress warnings from Python and external libraries
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

In [ ]:
fs_UiS4ADL = 100 # sampling frequency in Hz
path_UiS4ADL = f"../Data/UiS4ADL/Processed/UiS4ADL_{fs_UiS4ADL}hz_inactivity_removed.csv"
df = pd.read_csv(path_UiS4ADL)

fs_UiS4ADL_trimmed = 100 # sampling frequency in Hz
path_UiS4ADL_trimmed = f"../Data/UiS4ADL/Processed/UiS4ADL_{fs_UiS4ADL}hz.csv"
df_trimmed = pd.read_csv(path_UiS4ADL_trimmed)

fs_PAALADL = 32 # sampling frequency in Hz
path_PAALADL = f"../Data/PAAL_ADL/Processed/PAAL_ADL_{fs_PAALADL}hz.csv"
df_PAAL = pd.read_csv(path_PAALADL)

In [ ]:
with open('../Data/adl_dict.json', 'r') as f:
    adl_dict = json.load(f) # load the adl dictionary
adl_dict

{'1': 'drink water',
 '2': 'eat meal',
 '3': 'open a bottle',
 '4': 'open a box',
 '5': 'brush teeth',
 '6': 'brush hair',
 '7': 'take off jacket',
 '8': 'put on jacket',
 '9': 'put on shoe',
 '10': 'take off shoe',
 '11': 'put on glasses',
 '12': 'take off glasses',
 '13': 'sit down',
 '14': 'stand up',
 '15': 'writing',
 '16': 'phone call',
 '17': 'type on keyboard',
 '18': 'salute (wave hand)',
 '19': 'sneeze cough',
 '20': 'blow nose',
 '21': 'washing hands',
 '22': 'dusting',
 '23': 'ironing',
 '24': 'washing dishes'}

In [ ]:
# Select the ADL and the column to plot
adl = 4
df_adl_subset = df[df['adl'] == adl]
column_to_plot = 'fxLinAccX[g]'

## Statistics

In [ ]:
def compute_bounds(group):
    Q1 = group['duration'].quantile(0.25)
    Q3 = group['duration'].quantile(0.75)
    min_duration = group['duration'].min()
    max_duration = group['duration'].max()
    
    lower_bound = group['duration'].quantile(0.01)
    upper_bound = group['duration'].quantile(0.99)
    
    # Count how many fileIDs are below the lower bound
    below_lb = group[group['duration'] < lower_bound]['fileID'].nunique()
    # Count how many fileIDs are above the upper bound
    above_ub = group[group['duration'] > upper_bound]['fileID'].nunique()
    
    # Count how many fileIDs are below Q1
    below_Q1 = group[group['duration'] < Q1]['fileID'].nunique()
    # Count how many fileIDs are above Q3
    above_Q3 = group[group['duration'] > Q3]['fileID'].nunique()
    
    # Count how many fileIDs are below 1 second
    below_one_sec = group[group['duration'] < 1.0]['fileID'].nunique()
    
    # Count how many fileIDs are below 4 second
    below_four_sec = group[group['duration'] < 4]['fileID'].nunique()
    
    # Total number of fileIDs
    total_fileIDs = group['fileID'].nunique()
    
    # Percentage of fileIDs below lower bound
    per_below_lb = (below_lb / total_fileIDs) * 100
    # Percentage of fileIDs above upper bound
    per_above_ub = (above_ub / total_fileIDs) * 100
    
    # Percentage of fileIDs below 1 second
    per_below_one_sec = (below_one_sec / total_fileIDs) * 100
    
    # Percentage of fileIDs below 4 second
    per_below_four_sec = (below_four_sec / total_fileIDs) * 100

    return pd.Series({
        'min': min_duration,
        'Q1': Q1,
        'Q3': Q3,
        'max': max_duration,
        'lower_bound': lower_bound,
        'upper_bound': upper_bound,
        'f_below_lb': below_lb,
        'f_above_ub': above_ub,
        'f_below_Q1': below_Q1,
        'f_above_Q3': above_Q3,
        'f_below_one_sec': below_one_sec,
        'total_fileIDs': total_fileIDs,
        '%_files_below_lb': per_below_lb,
        '%_files_above_ub': per_above_ub,
        '%_files_below_one_sec': per_below_one_sec,
        '%_files_below_four_sec': per_below_four_sec
    })

### UiS4ADL before trimming

In [ ]:
df_copy = df.copy()

# Calculate the number of samples for each 'adl' and 'fileID'
df_copy['sample_count'] = df_copy.groupby(['adl', 'fileID'])['timestamp'].transform('count')

# Calculate the duration in seconds
df_copy['duration'] = (df_copy['sample_count'] * 10) / 1000

In [ ]:
# Apply the function to each 'adl' group
iqr_bounds = df_copy.groupby('adl').apply(compute_bounds).reset_index()

# Ensure 'adl' is a string to match the keys in adl_dict
iqr_bounds['adl'] = iqr_bounds['adl'].astype(str)

iqr_bounds['adl_name'] = iqr_bounds['adl'].map(adl_dict)

iqr_bounds

,adl,min,Q1,Q3,max,lower_bound,upper_bound,f_below_lb,f_above_ub,f_below_Q1,f_above_Q3,f_below_one_sec,total_fileIDs,%_files_below_lb,%_files_above_ub,%_files_below_one_sec,%_files_below_four_sec,adl_name
0,1,1.92,13.48,31.38,67.90,3.10,67.90,10.0,0.0,129.0,17.0,0.0,203.0,4.926108,0.00000,0.000000,18.226601,drink water
1,2,1.42,30.76,59.25,70.60,20.69,70.60,4.0,0.0,60.0,25.0,0.0,154.0,2.597403,0.00000,0.000000,0.649351,eat meal
2,3,1.56,10.16,31.05,55.81,2.96,55.81,11.0,0.0,128.0,19.0,0.0,210.0,5.238095,0.00000,0.000000,17.619048,open a bottle
3,4,1.30,11.80,30.41,72.64,2.04,72.64,12.0,0.0,138.0,17.0,0.0,209.0,5.741627,0.00000,0.000000,35.885167,open a box
4,5,10.24,20.91,60.20,66.24,12.20,64.89,6.0,1.0,91.0,27.0,0.0,210.0,2.857143,0.47619,0.000000,0.000000,brush teeth
5,6,4.25,16.01,58.99,64.68,8.74,64.68,8.0,0.0,122.0,21.0,0.0,215.0,3.720930,0.00000,0.000000,0.000000,brush hair
6,7,1.87,17.68,31.39,98.87,3.44,98.87,7.0,0.0,92.0,16.0,0.0,157.0,4.458599,0.00000,0.000000,12.101911,take off jacket
7,8,2.64,19.04,31.53,99.72,3.51,99.72,7.0,0.0,87.0,14.0,0.0,151.0,4.635762,0.00000,0.000000,8.609272,put on jacket
8,9,1.77,12.69,29.72,45.35,4.38,45.35,7.0,0.0,82.0,19.0,0.0,154.0,4.545455,0.00000,0.000000,3.246753,put on shoe
9,10,1.81,13.99,29.74,40.55,2.89,40.55,8.0,0.0,89.0,17.0,0.0,150.0,5.333333,0.00000,0.000000,14.000000,take off shoe


### UiS4ADL after trimming

In [ ]:
df_trimmed_copy = df_trimmed.copy()

# Calculate the number of samples for each 'adl' and 'fileID'
df_trimmed_copy['sample_count'] = df_trimmed_copy.groupby(['adl', 'fileID'])['timestamp'].transform('count')

# Calculate the duration in seconds
df_trimmed_copy['duration'] = (df_trimmed_copy['sample_count'] * 10) / 1000

In [ ]:
# Apply the function to each 'adl' group
iqr_trimmed_bounds = df_trimmed_copy.groupby('adl').apply(compute_bounds).reset_index()

# Ensure 'adl' is a string to match the keys in adl_dict
iqr_trimmed_bounds['adl'] = iqr_trimmed_bounds['adl'].astype(str)

iqr_trimmed_bounds['adl_name'] = iqr_trimmed_bounds['adl'].map(adl_dict)

iqr_trimmed_bounds

,adl,min,Q1,Q3,max,lower_bound,upper_bound,f_below_lb,f_above_ub,f_below_Q1,f_above_Q3,f_below_one_sec,total_fileIDs,%_files_below_lb,%_files_above_ub,%_files_below_one_sec,%_files_below_four_sec,adl_name
0,1,1.18,15.64,30.68,61.86,2.22,61.86,14.0,0.0,139.0,15.0,0.0,203.0,6.896552,0.0,0.000000,34.482759,drink water
1,2,1.42,29.68,57.50,68.04,18.58,68.04,4.0,0.0,63.0,24.0,0.0,154.0,2.597403,0.0,0.000000,0.649351,eat meal
2,3,1.12,10.70,28.84,53.52,1.54,53.52,17.0,0.0,147.0,16.0,0.0,210.0,8.095238,0.0,0.000000,48.095238,open a bottle
3,4,1.04,20.66,29.08,70.92,1.30,70.92,14.0,0.0,154.0,14.0,0.0,209.0,6.698565,0.0,0.000000,56.937799,open a box
4,5,3.02,18.62,56.64,60.92,10.50,60.92,7.0,0.0,95.0,25.0,0.0,210.0,3.333333,0.0,0.000000,0.476190,brush teeth
5,6,1.54,14.38,57.54,63.16,6.30,63.16,10.0,0.0,132.0,20.0,0.0,215.0,4.651163,0.0,0.000000,1.860465,brush hair
6,7,1.08,15.00,28.96,93.90,1.78,93.90,13.0,0.0,100.0,12.0,0.0,157.0,8.280255,0.0,0.000000,37.579618,take off jacket
7,8,1.08,17.82,28.20,97.06,2.54,97.06,9.0,0.0,94.0,11.0,0.0,151.0,5.960265,0.0,0.000000,27.152318,put on jacket
8,9,1.64,10.02,25.42,43.58,2.20,43.58,9.0,0.0,89.0,16.0,0.0,154.0,5.844156,0.0,0.000000,19.480519,put on shoe
9,10,1.04,11.46,25.48,38.44,1.80,38.44,11.0,0.0,97.0,13.0,0.0,150.0,7.333333,0.0,0.000000,38.666667,take off shoe


### PAAL ADL

In [ ]:
df_PAAL_copy = df_PAAL.copy()

# Calculate the number of samples for each 'adl' and 'fileID'
df_PAAL_copy['sample_count'] = df_PAAL_copy.groupby(['adl', 'fileID'])['timestamp'].transform('count')

# Calculate the duration in seconds
df_PAAL_copy['duration'] = df_PAAL_copy['sample_count'] / fs_PAALADL

In [ ]:
# Apply the function to each 'adl' group
iqr_PAAL_bounds = df_PAAL_copy.groupby('adl').apply(compute_bounds).reset_index()

# Ensure 'adl' is a string to match the keys in adl_dict
iqr_PAAL_bounds['adl'] = iqr_PAAL_bounds['adl'].astype(str)

iqr_PAAL_bounds['adl_name'] = iqr_PAAL_bounds['adl'].map(adl_dict)

iqr_PAAL_bounds

,adl,min,Q1,Q3,max,lower_bound,upper_bound,f_below_lb,f_above_ub,f_below_Q1,f_above_Q3,f_below_one_sec,total_fileIDs,%_files_below_lb,%_files_above_ub,%_files_below_one_sec,%_files_below_four_sec,adl_name
0,1,1.15625,3.15625,5.53125,11.46875,1.75000,11.468750,6.0,0.0,98.0,34.0,0.0,260.0,2.307692,0.000000,0.000000,67.692308,drink water
1,2,0.43750,2.18750,4.18750,10.75000,0.75000,10.750000,3.0,0.0,111.0,28.0,25.0,252.0,1.190476,0.000000,9.920635,85.714286,eat meal
2,3,0.21875,1.78125,3.71875,8.12500,0.62500,8.125000,10.0,0.0,119.0,27.0,45.0,255.0,3.921569,0.000000,17.647059,90.196078,open a bottle
3,4,0.31250,1.46875,3.31250,12.75000,0.53125,12.750000,6.0,0.0,118.0,19.0,58.0,246.0,2.439024,0.000000,23.577236,95.934959,open a box
4,5,2.53125,13.68750,19.75000,41.75000,3.84375,41.750000,8.0,0.0,91.0,35.0,0.0,227.0,3.524229,0.000000,0.000000,5.286344,brush teeth
5,6,1.37500,10.87500,18.43750,254.62500,3.12500,254.625000,13.0,0.0,116.0,23.0,0.0,246.0,5.284553,0.000000,0.000000,8.536585,brush hair
6,7,1.65625,4.46875,8.40625,15.62500,2.28125,15.625000,5.0,0.0,98.0,33.0,0.0,253.0,1.976285,0.000000,0.000000,30.039526,take off jacket
7,8,1.87500,6.46875,13.84375,24.00000,3.62500,24.000000,5.0,0.0,108.0,32.0,0.0,253.0,1.976285,0.000000,0.000000,4.347826,put on jacket
8,9,1.56250,8.31250,14.12500,28.90625,2.40625,28.906250,11.0,0.0,119.0,33.0,0.0,256.0,4.296875,0.000000,0.000000,17.968750,put on shoe
9,10,0.62500,3.65625,8.84375,26.37500,1.12500,26.375000,11.0,0.0,118.0,22.0,8.0,244.0,4.508197,0.000000,3.278689,52.868852,take off shoe
